In [0]:
dbutils.widgets.text("api_end_point", "https://dummyjson.com/products")
dbutils.widgets.text("storage_name", "miniproject123")
dbutils.widgets.text("key_scope", "secret-scope")

In [0]:
api_endpoint = dbutils.widgets.get("api_end_point")
storage_name = dbutils.widgets.get("storage_name")
key_scope = dbutils.widgets.get("key_scope")
tenant_id     = dbutils.secrets.get(scope=key_scope, key="client-tenant")
client_id     = dbutils.secrets.get(scope=key_scope, key="client-secret")
client_secret = dbutils.secrets.get(scope=key_scope, key="client-value")

spark.conf.set(f"fs.azure.account.auth.type.{storage_name}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_name}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_name}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_name}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_name}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

In [0]:
import requests
from requests.adapters import HTTPAdapter
from urllib3.util import Retry
from pyspark.sql import functions as F

def fetch_api_data(url:str):
    retry = Retry(
        total = 3,
        backoff_factor = 2,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"]
    )
    adapter = HTTPAdapter(max_retries=retry)
    session = requests.Session()
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    # Add User-Agent header to avoid 403 Forbidden
    
    try:
        response = session.get(url, timeout = 10)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"Failed to fetch data from API: {e}")
        raise e
products_data = fetch_api_data(api_endpoint)



In [0]:
from pyspark.sql import functions as F
products_list = products_data["products"]
import json
rdd = spark.sparkContext.parallelize([json.dumps(r) for r in products_list])
df_products = spark.read.json(rdd) \
    .withColumn("ingestion_timestamp", F.current_timestamp()) \
    .withColumn("source_system", F.lit("dummyjson_products_api"))

# 4. Save to ADLS Raw Path in Delta Format
bronze_path = f"abfss://bronze@{storage_name}.dfs.core.windows.net/ingested_products/"

df_products.write \
    .mode("overwrite") \
    .format("delta") \
    .save(bronze_path)